# 03 Facets Reviews

Computes review-side diversity facets M0-M5 over `condition × text_version × field`. Reviews are analyzed as paired, per-target-proposal exact-n panels using the row-index cache written by `4b`. M6 is not applicable to reviews.

In [1]:
CONFIG = dict(
    conditions=["baseline", "one_at_a_time", "persona"],
    text_versions=["rephrased", "original"],
    fields=["whole", "strengths", "weakness"],
    models=["claude", "gemini", "gpt"],
    n_human=23,
    seed=42,
    B_perm=10_000,
    B_sub=1_000,
)

RUN_OT = True
WRITE_FIGURES = True

## Load

Load review artifacts, assert row-order contracts, and convert exact-n row-index panels into the UID-combination shape expected by the review facet helper.

In [2]:
import json
import pickle
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import diversity_inference as di
import plotting as pl
from compare_review_diversity import load_pickle, load_review_analysis_inputs

if not RUN_OT:
    di.wasserstein_ot = lambda X, Y: np.nan

TEST_COLUMNS = [
    "condition", "task", "text_version", "field", "comparison", "facet", "metric", "is_primary", "param",
    "human_value", "ai_value", "effect_size", "effect_type", "ci_lo", "ci_hi", "human_ci_lo", "human_ci_hi",
    "inference", "stat", "p_raw", "p_fdr", "n_human", "n_ai", "n_perm_or_sub", "parity_ref", "notes",
]
PRIMARY = {
    ("spread", "mean_pairwise", ""),
    ("richness", "vendi", "q=1"),
    ("coverage", "coverage_geometric", "k=panel_adaptive"),
    ("dimensionality", "participation_ratio", ""),
    ("evenness", "ripley_excess", "r=pooled_q01_q50"),
    ("displacement", "mmd2", ""),
}

def _load_json(path):
    return json.loads(Path(path).read_text())

def _standardize_tests(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    if "n_perm_or_boot" in out.columns:
        out = out.rename(columns={"n_perm_or_boot": "n_perm_or_sub"})
    for col in TEST_COLUMNS:
        if col not in out.columns:
            out[col] = np.nan if col not in {"field", "param", "notes"} else ""
    out["param"] = out["param"].fillna("")
    out["is_primary"] = out.apply(lambda r: (r["facet"], r["metric"], r["param"]) in PRIMARY, axis=1)
    out.loc[out["facet"].ne("coverage"), "parity_ref"] = 1.0
    # Review coverage parity = the mean human LOO self-coverage exported as human_value
    # (finite-sample same-panel reference), NOT 1.0 (spec 1A.6 / 15.2).
    cov_mask = out["metric"].eq("coverage_geometric")
    out.loc[cov_mask, "parity_ref"] = out.loc[cov_mask, "human_value"]
    return out[TEST_COLUMNS]

def _standardize_gradient(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"JT", "p_raw", "p_fdr"} else ""
    return out[["condition", "task", "text_version", "field", "facet", "metric", "param", "order", "JT", "p_raw", "p_fdr", "direction_ok", "notes"]]

def _standardize_curves(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "task", "text_version", "field", "group", "facet", "metric", "param", "x", "y", "y_lo", "y_hi"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"x", "y", "y_lo", "y_hi"} else ""
    out["param"] = out["param"].fillna("")
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    return out[["condition", "task", "text_version", "field", "group", "facet", "metric", "param", "x", "y", "y_lo", "y_hi"]]

def _standardize_paired(df):
    out = df.copy()
    if "text_branch" in out.columns:
        out = out.rename(columns={"text_branch": "text_version"})
    for col in ["condition", "text_version", "field", "comparison", "facet", "metric", "param", "target_proposal_uid", "target_cohort", "n_human_reviews", "human_value", "ai_value", "paired_diff"]:
        if col not in out.columns:
            out[col] = np.nan if col in {"n_human_reviews", "human_value", "ai_value", "paired_diff"} else ""
    return out[["condition", "text_version", "field", "comparison", "facet", "metric", "param", "target_proposal_uid", "target_cohort", "n_human_reviews", "human_value", "ai_value", "paired_diff"]]

def _assert_review_contract(condition, text_version):
    prep_dir = PROJECT_ROOT / "data" / "prepared" / condition / "reviews" / text_version
    manifest = _load_json(prep_dir / "prepare_manifest.json")
    master = pd.read_csv(prep_dir / "review_master.csv")
    assert manifest["embeddings_l2_normalized"] is True, f"run modified 4b first: {condition}/{text_version}"
    assert manifest["review_uid_order"] == master["review_uid"].astype(str).tolist(), "row order drift in review master"
    assert "source_family" in master.columns, "review model family column missing: expected source_family"
    return prep_dir, manifest, master

def _panels_to_combination_cache(master, panels):
    cache = {"all_ai": {}, "claude": {}, "gemini": {}, "gpt": {}}
    review_uids = master["review_uid"].astype(str).tolist()
    for uid, payload in panels.items():
        human_uids = [review_uids[int(i)] for i in payload["human_idx"]]
        cache["all_ai"][uid] = {
            "eligible": True,
            "human_review_uids": human_uids,
            "ai_panel_combinations": [[review_uids[int(i)] for i in combo] for combo in payload["pooled"]],
        }
        for model in ["claude", "gemini", "gpt"]:
            cache[model][uid] = {
                "eligible": True,
                "human_review_uids": human_uids,
                "ai_panel_combinations": [[review_uids[int(i)] for i in combo] for combo in payload["per_model"].get(model, [])],
            }
    return cache

def _analysis_for_field(condition, text_version, field, manifest):
    analysis = load_review_analysis_inputs(PROJECT_ROOT, condition, text_version=text_version)
    if field == "whole":
        return analysis
    field_key = f"{field}_embedding_path"
    if field_key not in manifest:
        return None
    bundle = load_pickle(Path(manifest[field_key]))
    return replace(analysis, review_embeddings=bundle)

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/eveyhuang/Documents/NICO/human-AI-proposal


## Panels

Use `review_panels_exact_n.pkl` from `4b`; indices are positional and checked against `review_uid_order` before conversion.

In [3]:
all_tests = []
all_gradients = []
all_curves = []
all_paired = []

for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        prep_dir, manifest, master = _assert_review_contract(condition, text_version)
        with open(manifest["review_panels_exact_n_file"], "rb") as fh:
            panels = pickle.load(fh)
        combination_cache = _panels_to_combination_cache(master, panels)
        for field in CONFIG["fields"]:
            if field not in manifest.get("fields_available", ["whole"]):
                print(f"Skipping unavailable field: {condition}/{text_version}/{field}")
                continue
            print(f"\n=== Reviews: {condition}/{text_version}/{field} ===")
            analysis = _analysis_for_field(condition, text_version, field, manifest)
            if analysis is None:
                continue
            tests, gradients, curves, paired = di.build_review_facet_outputs(
                analysis,
                combination_cache,
                text_branch=text_version,
                field=field,
                n_boot=CONFIG["B_sub"],
                n_perm=CONFIG["B_perm"],
                seed=CONFIG["seed"],
            )
            all_tests.append(_standardize_tests(tests))
            all_gradients.append(_standardize_gradient(gradients))
            all_curves.append(_standardize_curves(curves))
            all_paired.append(_standardize_paired(paired))

review_tests_df = pd.concat(all_tests, ignore_index=True) if all_tests else pd.DataFrame(columns=TEST_COLUMNS)
review_gradient_df = pd.concat(all_gradients, ignore_index=True) if all_gradients else pd.DataFrame()
review_curves_df = pd.concat(all_curves, ignore_index=True) if all_curves else pd.DataFrame()
review_paired_df = pd.concat(all_paired, ignore_index=True) if all_paired else pd.DataFrame()

# Family FDR: BH over the secondary family within each (task, text_version, field),
# across conditions and comparisons; primaries stay on p_raw (spec 1.7).
review_tests_df = di.apply_family_fdr(review_tests_df)
if not review_gradient_df.empty:
    review_gradient_df["p_fdr"] = np.nan
    for _, idx in review_gradient_df.groupby(["task", "text_version"]).groups.items():
        idx = list(idx)
        review_gradient_df.loc[idx, "p_fdr"] = di.benjamini_hochberg(review_gradient_df.loc[idx, "p_raw"])
review_tests_df.head()


=== Reviews: baseline/rephrased/whole ===

=== Reviews: baseline/rephrased/strengths ===

=== Reviews: baseline/rephrased/weakness ===

=== Reviews: baseline/original/whole ===
Skipping unavailable field: baseline/original/strengths
Skipping unavailable field: baseline/original/weakness

=== Reviews: one_at_a_time/rephrased/whole ===

=== Reviews: one_at_a_time/rephrased/strengths ===

=== Reviews: one_at_a_time/rephrased/weakness ===

=== Reviews: one_at_a_time/original/whole ===
Skipping unavailable field: one_at_a_time/original/strengths
Skipping unavailable field: one_at_a_time/original/weakness

=== Reviews: persona/rephrased/whole ===

=== Reviews: persona/rephrased/strengths ===

=== Reviews: persona/rephrased/weakness ===

=== Reviews: persona/original/whole ===
Skipping unavailable field: persona/original/strengths
Skipping unavailable field: persona/original/weakness


,condition,task,text_version,field,comparison,facet,metric,is_primary,param,human_value,...,human_ci_hi,inference,stat,p_raw,p_fdr,n_human,n_ai,n_perm_or_sub,parity_ref,notes
0,baseline,reviews,rephrased,whole,human_vs_pooled_ai,richness,vendi,True,q=1,1.183899,...,NaN,paired_wilcoxon,82.0,0.091753,NaN,3.695652,3.695652,23,1.000000,paired across target proposals; AI value = mea...
1,baseline,reviews,rephrased,whole,human_vs_pooled_ai,dimensionality,participation_ratio,True,,2.515631,...,NaN,paired_wilcoxon,105.0,0.485170,NaN,3.695652,3.695652,23,1.000000,paired across target proposals; AI value = mea...
2,baseline,reviews,rephrased,whole,human_vs_pooled_ai,dimensionality,effective_rank,False,,2.598130,...,NaN,paired_wilcoxon,106.0,0.505701,0.602685,3.695652,3.695652,23,1.000000,paired across target proposals; AI value = mea...
3,baseline,reviews,rephrased,whole,human_vs_pooled_ai,coverage,coverage_geometric,True,k=panel_adaptive,0.829545,...,NaN,paired_wilcoxon,23.0,0.003759,NaN,3.695652,3.695652,22,0.829545,paired across target proposals; AI value = mea...
4,baseline,reviews,rephrased,whole,human_vs_pooled_ai,evenness,vendi_slope,False,q=0..2,0.701374,...,NaN,paired_wilcoxon,74.0,0.052230,0.068848,3.695652,3.695652,23,1.000000,paired across target proposals; AI value = mea...


## Export

Write paired review facet outputs per `{condition}/reviews/{text_version}` and cross-condition copies.

In [4]:
def _write_curves(path, df):
    out = df.copy()
    for numeric_col in ["x", "y", "y_lo", "y_hi"]:
        if numeric_col in out.columns:
            out[numeric_col] = pd.to_numeric(out[numeric_col], errors="coerce")
    try:
        out.to_parquet(path, index=False)
    except Exception as exc:
        raise RuntimeError(f"Could not write required parquet {path}: {exc}") from exc


# Review fingerprint (redesign spec 3.2): sign-aligned Cliff's delta per facet, so on
# every row LEFT of 0 = AI panels less diverse. Clumping metrics enter negated.
REVIEW_FP_SPECS = [
    ("spread", "mean_pairwise", "", +1),
    ("richness", "vendi", "q=1", +1),
    ("evenness", "ripley_excess", "r=pooled_q01_q50", -1),
    ("dimensionality", "participation_ratio", "", +1),
    ("coverage", "coverage_geometric", "k=panel_adaptive", +1),
]

def _fingerprint_rows(condition, text_version):
    tests = review_tests_df[(review_tests_df.condition == condition)
                            & (review_tests_df.text_version == text_version)
                            & review_tests_df.field.eq("whole")]
    out = []
    for facet, metric, param, sign in REVIEW_FP_SPECS:
        rows = tests[tests.facet.eq(facet) & tests.metric.eq(metric) & tests.param.eq(param)]
        for comparison, group in [("human_vs_claude", "Claude"), ("human_vs_gemini", "Gemini"),
                                  ("human_vs_gpt", "GPT"), ("human_vs_pooled_ai", "All AI")]:
            r = rows[rows.comparison.eq(comparison)]
            if r.empty:
                continue
            r = r.iloc[0]
            lo, hi = sign * r["ci_lo"], sign * r["ci_hi"]
            if np.isfinite(lo) and np.isfinite(hi) and lo > hi:
                lo, hi = hi, lo
            out.append({"condition": condition, "task": "reviews", "text_version": text_version, "field": "whole",
                        "facet": facet, "metric": metric, "param": param, "group": group,
                        "value": r["effect_size"], "z": sign * r["effect_size"],
                        "z_ci_lo": lo, "z_ci_hi": hi, "sign_aligned": sign,
                        "stars": pl.row_stars(r), "mode": "delta"})
    return pd.DataFrame(out)


def _write_cell_outputs(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "reviews" / text_version
    figures_dir = PROJECT_ROOT / "results" / "figures" / condition / "reviews" / text_version
    tables_dir.mkdir(parents=True, exist_ok=True)
    figures_dir.mkdir(parents=True, exist_ok=True)
    tests = review_tests_df[(review_tests_df.condition == condition) & (review_tests_df.text_version == text_version)].copy()
    gradients = review_gradient_df[(review_gradient_df.condition == condition) & (review_gradient_df.text_version == text_version)].copy()
    curves = review_curves_df[(review_curves_df.condition == condition) & (review_curves_df.text_version == text_version)].copy()
    paired = review_paired_df[(review_paired_df.condition == condition) & (review_paired_df.text_version == text_version)].copy()
    tests.to_csv(tables_dir / "facet_diversity_tests.csv", index=False)
    gradients.to_csv(tables_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(tables_dir / "facet_diversity_curves.parquet", curves)
    paired.to_csv(tables_dir / "facet_review_paired_long.csv", index=False)
    _fingerprint_rows(condition, text_version).to_csv(tables_dir / "facet_fingerprint.csv", index=False)
    return tables_dir, figures_dir

written = []
for condition in CONFIG["conditions"]:
    for text_version in CONFIG["text_versions"]:
        written.append((condition, text_version, *_write_cell_outputs(condition, text_version)))

for text_version in CONFIG["text_versions"]:
    cross_dir = PROJECT_ROOT / "results" / "tables" / "cross_condition" / "reviews" / text_version
    cross_dir.mkdir(parents=True, exist_ok=True)
    review_tests_df[review_tests_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_tests.csv", index=False)
    review_gradient_df[review_gradient_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_diversity_gradient.csv", index=False)
    _write_curves(cross_dir / "facet_diversity_curves.parquet", review_curves_df[review_curves_df.text_version.eq(text_version)])
    review_paired_df[review_paired_df.text_version.eq(text_version)].to_csv(cross_dir / "facet_review_paired_long.csv", index=False)

pd.DataFrame(written, columns=["condition", "text_version", "tables_dir", "figures_dir"])

,condition,text_version,tables_dir,figures_dir
0,baseline,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
1,baseline,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
2,one_at_a_time,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
3,one_at_a_time,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
4,persona,rephrased,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...
5,persona,original,/Users/eveyhuang/Documents/NICO/human-AI-propo...,/Users/eveyhuang/Documents/NICO/human-AI-propo...


## Figures

Real §12.7 views from the tidy tables: paired boxes with funded-target rings, an equal-n whole-cloud ridge/profile/scree/envelope/CDF set (each group subsampled to the human review count — the curves are n-sensitive), Cliff's-δ forest effect panels (AI − Human with bootstrap CIs), the per-proposal paired coverage scatter against the leave-one-out human reference, the pooled-cloud MMD² bar (labeled unpaired/exploratory), a palette-correct review-space UMAP, and one paired-slope figure per primary metric (one line per proposal). Saved as PNG + PDF.

In [5]:
METRIC_LABELS = {
    ("spread", "mean_pairwise"): "mean pairwise cosine distance",
    ("richness", "vendi"): "Vendi VS₁ (effective distinct reviews)",
    ("coverage", "coverage_geometric"): "coverage of the human review span",
    ("dimensionality", "participation_ratio"): "participation ratio",
    ("evenness", "ripley_excess"): "Ripley excess area vs matched-size null",
    ("evenness", "vendi_slope"): "Vendi profile drop (VS₀−VS₂)/VS₀",
}


def _review_convergent_box(paired, out_base, title):
    """Dodged paired-diff boxes for the five convergent M0 metrics, z-scored within metric."""
    metrics = ["centroid_loo", "mst_dispersion", "sparseness", "nn_isolation", "spherical_variance"]
    sub = paired[paired["facet"].eq("spread") & paired["metric"].isin(metrics)]
    fig, ax = plt.subplots(figsize=(11, 5))
    if sub.empty:
        ax.text(0.5, 0.5, "No convergent spread rows", ha="center", va="center")
        ax.set_axis_off()
        pl.save_fig(fig, out_base)
        return
    comps = ["human_vs_claude", "human_vs_gemini", "human_vs_gpt", "human_vs_pooled_ai"]
    width = 0.19
    for mi, metric in enumerate(metrics):
        vals_all = sub.loc[sub["metric"].eq(metric), "paired_diff"].to_numpy(dtype=float)
        mu, sd = np.nanmean(vals_all), np.nanstd(vals_all)
        sd = sd if sd > 0 else 1.0
        for gi, comp in enumerate(comps):
            vals = sub.loc[sub["metric"].eq(metric) & sub["comparison"].eq(comp), "paired_diff"].dropna().to_numpy(dtype=float)
            if vals.size == 0:
                continue
            g = pl.COMPARISON_TO_GROUP[comp]
            bp = ax.boxplot([(vals - mu) / sd], positions=[mi + (gi - 1.5) * width], widths=width * 0.9, patch_artist=True)
            bp["boxes"][0].set_facecolor(pl.PALETTE[g])
            bp["boxes"][0].set_alpha(0.7)
            bp["medians"][0].set_color("black")
    ax.axhline(0.0, color="#404040", linestyle="--", linewidth=1)
    ax.set_xticks(range(len(metrics)), metrics, rotation=15)
    ax.set_ylabel("paired Human − AI (z-scored within metric)\n↑ humans more diverse")
    ax.set_title(title)
    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=pl.PALETTE[pl.COMPARISON_TO_GROUP[c]], alpha=0.7, edgecolor="black") for c in comps]
    ax.legend(handles, [pl.COMPARISON_TO_GROUP[c] for c in comps], fontsize=8, ncols=4)
    pl.add_direction_badge(fig, "↑ humans more diverse (paired)")
    fig.text(0.01, -0.02, "Points behind each box = 23 paired target-proposal differences (Human panel − AI exact-n panel mean); "
                          "z-scored within metric for a shared axis. One facet, several views (spec 3.1).",
             ha="left", va="top", fontsize=7.5, color="#333333")
    pl.save_fig(fig, out_base)


def emit_required_figures(condition, text_version):
    tables_dir = PROJECT_ROOT / "results" / "tables" / condition / "reviews" / text_version
    figs = PROJECT_ROOT / "results" / "figures" / condition / "reviews" / text_version
    tests = pd.read_csv(tables_dir / "facet_diversity_tests.csv")
    paired = pd.read_csv(tables_dir / "facet_review_paired_long.csv")
    curves = pd.read_parquet(tables_dir / "facet_diversity_curves.parquet")
    for df in (tests, paired, curves):
        if "param" in df.columns:
            df["param"] = df["param"].fillna("")
    tests = tests[tests["field"].eq("whole")]
    paired = paired[paired["field"].eq("whole")]
    curves = curves[curves["field"].eq("whole")]
    fig_ctx = f"{condition} · reviews/{text_version}"

    prep = PROJECT_ROOT / "data" / "prepared" / condition / "reviews" / text_version
    master = pd.read_csv(prep / "review_master.csv")
    funded_targets = set(master.loc[master["target_funding"].astype(str).str.lower().eq("true"),
                                    "target_proposal_uid"].astype(str))

    # Fingerprint (redesign spec 3.2): sign-aligned Cliff's delta, one shared axis.
    fingerprint = pd.read_csv(tables_dir / "facet_fingerprint.csv")
    fingerprint["param"] = fingerprint["param"].fillna("")
    pl.plot_fingerprint(fingerprint, figs / "facet_fingerprint", mode="delta",
                        title=f"Diversity fingerprint — paired effects, one axis · {fig_ctx}")

    def paired_box(stem, facet, metric):
        pl.plot_paired_box(paired, figs / stem, title=f"{facet.title()} — {metric} · {fig_ctx}",
                           facet=facet, metric=metric,
                           ylabel=f"paired Human − AI: {METRIC_LABELS.get((facet, metric), metric)}",
                           funded_targets=funded_targets)

    def effect(stem, facet, metric, param):
        pl.plot_effect_delta(tests, figs / stem, title=f"{facet.title()} — {metric} · {fig_ctx}",
                             facet=facet, metric=metric, param=param)

    # Spread (M0)
    paired_box("spread_mean_pairwise_box", "spread", "mean_pairwise")
    pl.plot_pairwise_ridge(curves, figs / "spread_mean_pairwise_ridge",
                           title=f"Spread — mean_pairwise, whole-cloud distributions (equal-n) · {fig_ctx}")
    effect("spread_mean_pairwise_effect", "spread", "mean_pairwise", "")
    _review_convergent_box(paired, figs / "spread_convergent_box", f"Spread — convergent metrics · {fig_ctx}")

    # Richness (M1)
    pl.plot_vendi_profile(curves, figs / "richness_vendi_profile",
                          title=f"Richness — Vendi profile, whole-cloud (equal-n) · {fig_ctx}")
    pl.plot_scree(curves, figs / "richness_vendi_scree", metric="kernel_eigen_scree",
                  title=f"Richness — Vendi eigenvalue scree (equal-n, diagnostic) · {fig_ctx}", xlabel="eigen-index",
                  ylabel="normalized kernel eigenvalue (cliff = few dominant modes = LESS diverse)",
                  log_y=True, max_x=30, badge="↓ steeper cliff = less diverse")
    paired_box("richness_vendi_box", "richness", "vendi")
    effect("richness_vendi_effect", "richness", "vendi", "q=1")

    # Evenness (M4 + vendi_slope) — re-oriented per redesign spec 2.
    pl.plot_ripley_envelope(curves, figs / "evenness_ripley_excess_envelope",
                            title=f"Evenness — vs same-size null, whole-cloud (equal-n) · {fig_ctx}")
    pl.plot_g_cdf(curves, figs / "evenness_g_function_cdf",
                  title=f"Evenness — twin-free fraction (1−G), whole-cloud (equal-n) · {fig_ctx}")
    pl.plot_nn_distance_hist(curves, figs / "evenness_nn_similarity_hist",
                             title=f"Evenness — NN distances (equal-n draw) · {fig_ctx}")
    paired_box("evenness_vendi_slope_box", "evenness", "vendi_slope")
    paired_box("evenness_ripley_excess_box", "evenness", "ripley_excess")

    # Dimensionality (M3) — residual-variation orientation per redesign spec 2.
    pl.plot_scree(curves, figs / "dimensionality_participation_ratio_scree", metric="participation_ratio_scree",
                  title=f"Dimensionality — residual variation (equal-n) · {fig_ctx}",
                  xlabel="principal component index", ylabel="variance remaining beyond first x components",
                  mark_90=True, max_x=30, residual=True, badge="↑ higher-dimensional")
    paired_box("dimensionality_participation_ratio_box", "dimensionality", "participation_ratio")
    effect("dimensionality_participation_ratio_effect", "dimensionality", "participation_ratio", "")

    # Coverage (M2, panel-paired)
    pl.plot_review_coverage_paired_scatter(paired, figs / "coverage_geometric_scatter",
                                           title=f"Coverage — geometric, per-proposal paired · {fig_ctx}")
    paired_box("coverage_geometric_box", "coverage", "coverage_geometric")
    effect("coverage_geometric_effect", "coverage", "coverage_geometric", "k=panel_adaptive")

    # Displacement (M5 — quarantined directional check, redesign spec 1.2)
    pl.plot_mmd_bar(tests, figs / "displacement_mmd2_bar",
                    title=f"Displacement — MMD², pooled review clouds (unpaired, exploratory) · {fig_ctx}")

    # Review-space UMAP (illustration only)
    coords_path = prep / "review_umap2d.npy"
    if coords_path.exists():
        coords = np.load(coords_path)
        funded_mask = master["target_funding"].astype(str).str.lower().eq("true").to_numpy()
        pl.plot_group_umap(coords, master["source_group"], figs / "review_space_umap",
                           title=f"Review-space UMAP (illustration only) · {fig_ctx}",
                           funded_mask=funded_mask)

    # Paired-slope figures for every primary metric (spec 11.5 / 12.7).
    for facet, metric in [("spread", "mean_pairwise"), ("richness", "vendi"),
                          ("coverage", "coverage_geometric"), ("dimensionality", "participation_ratio"),
                          ("evenness", "ripley_excess")]:
        pl.plot_paired_slope(paired, figs / f"{facet}_{metric}_paired_slope",
                             title=f"{facet.title()} — {metric} paired slopes · {fig_ctx}",
                             facet=facet, metric=metric,
                             ylabel=METRIC_LABELS.get((facet, metric), metric),
                             funded_targets=funded_targets)


if WRITE_FIGURES:
    for condition in CONFIG["conditions"]:
        for text_version in CONFIG["text_versions"]:
            emit_required_figures(condition, text_version)
print("Review facet figures complete.")

Review facet figures complete.


## Cross-condition Ratios

Filtering-side persona-persistence panels (spec §12.6): AI÷Human ratio of the paired panel means per condition. Ratios use the same human reference exported in `human_value` (LOO self-coverage for M2), so parity is 1.0 in ratio space. `ripley_excess` is excluded (envelope-area effect, not a ratio).

In [6]:
CROSS_SPECS = [
    ("spread", "mean_pairwise", "", "Mean pairwise distance"),
    ("richness", "vendi", "q=1", "Vendi VS₁"),
    ("coverage", "coverage_geometric", "k=panel_adaptive", "Geometric coverage"),
    ("dimensionality", "participation_ratio", "", "Participation ratio"),
]

if WRITE_FIGURES:
    for text_version in CONFIG["text_versions"]:
        ccdir = PROJECT_ROOT / "results" / "figures" / "cross_condition" / "reviews" / text_version
        for facet, metric, param, label in CROSS_SPECS:
            pl.plot_cross_condition_ratio(
                review_tests_df, ccdir / f"{facet}_{metric}_ratio_by_condition",
                facet=facet, metric=metric, param=param, task="reviews", text_version=text_version,
                title=f"{facet.title()} — {label} · AI÷Human by condition · reviews/{text_version}")
print("Cross-condition review figures complete.")

Cross-condition review figures complete.
